# NB09 — Bonus: DishWipe Full-Body (Winner Only) — RTX 5090

Self-contained bonus notebook. Load the **winning method from NB08**,
train it on the **DishWipe Full-Body** task with RTX 5090 specs, then
evaluate **200 episodes** and compare cross-task performance.

| Item | Value |
|------|-------|
| Task | DishWipe Full-Body (bonus) |
| Method | Winner from NB08 (`best_method.json`) |
| Robot | Unitree G1 Full Body (37 DOF) |
| Budget | 2,000,000 env steps (RTX 5090) |
| Eval | 200 episodes, 50K bootstrap, Welch's t-test |


## Objectives

1. Load winner declaration from NB08.
2. Quick smoke test on DishWipe Full-Body env.
3. Train winner method on DishWipe (**2M steps**, RTX 5090 config).
4. Evaluate **200 deterministic episodes**.
5. Bootstrap 95% CI (50K resamples) + Welch's t-test vs Apple performance.
6. Cross-task comparison: Apple vs DishWipe.


## Resources

| Resource | Requirement | Notes |
|----------|-------------|-------|
| GPU | **RTX 5090** (32 GB VRAM) | Training |
| RAM | 40 GB | Large buffer/net |
| Runtime | ~2-4 hours | 2M steps |


## Imports & Setup


In [ ]:
import sys, os, json, random, time
from pathlib import Path

import numpy as np
import torch
import gymnasium as gym
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.envs import UnitreeG1DishWipeFullBodyEnv

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB09"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

IS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if IS_GPU else "cpu"
print(f"Device: {DEVICE}")


## Step 1 — Load Winner from NB08


In [ ]:
with open(PROJECT_ROOT / "artifacts" / "NB08" / "best_method.json") as f:
    winner_info = json.load(f)

winner_method = winner_info["winner"]
print(f"🏆 Winner from NB08: {winner_method}")
print(f"  Apple mean_reward:  {winner_info['mean_reward']:.4f}")
print(f"  Apple success_rate: {winner_info['success_rate']:.2%}")


## Configuration


In [ ]:
def linear_schedule(initial_lr: float, final_lr: float = 1e-5):
    """Linear LR decay from initial_lr → final_lr."""
    def schedule(progress_remaining: float) -> float:
        return final_lr + (initial_lr - final_lr) * progress_remaining
    return schedule


CFG = {
    "seed":             42,
    "env_id":           "UnitreeG1DishWipeFullBody-v1",
    "control_mode":     "pd_joint_delta_pos",
    "obs_mode":         "state",
    "winner_method":    winner_method,
    "total_env_steps":  2_000_000 if IS_GPU else 20_000,
    "eval_episodes":    200,
    "max_steps_per_ep": 1000,
    "bootstrap_n":      50_000,
    "checkpoint_freq":  200_000,
    # ── RTX 5090 hyperparams (same as NB05/NB06) ──
    "learning_rate":    3e-4,
    "lr_end":           1e-5,
    "net_arch":         [512, 512],
    "activation_fn":    "ReLU",
}

SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

with open(ARTIFACTS_DIR / "nb09_config.json", "w") as f:
    json.dump(CFG, f, indent=2)
print(f"Training {winner_method} on DishWipe for {CFG['total_env_steps']:,} steps")
print(f"  net_arch: {CFG['net_arch']}, activation: {CFG['activation_fn']}")
print(f"  LR: {CFG['learning_rate']} → {CFG['lr_end']} (linear decay)")


## Step 2 — DishWipe Full-Body Smoke Test


In [ ]:
print("--- DishWipe Full-Body Smoke Test ---")
env_test = gym.make(CFG["env_id"], num_envs=1, obs_mode="state",
                    control_mode=CFG["control_mode"], render_mode="rgb_array")
env_test = CPUGymWrapper(env_test)
obs, info = env_test.reset(seed=SEED)

print(f"  obs shape: {obs.shape}")
print(f"  act shape: {env_test.action_space.shape}")

for _ in range(5):
    action = env_test.action_space.sample()
    obs, reward, terminated, truncated, info = env_test.step(action)
print(f"  Smoke OK, reward sample: {reward:.4f}")
env_test.close()


## Step 3 — Helpers for Residual-SAC (if winner)


In [ ]:
# Define BaseController + ResidualActionWrapper in case Residual-SAC won
def heuristic_dishwipe_policy(obs, env):
    """Simple heuristic for DishWipe: arm joints get small sweeping signals."""
    action = np.zeros(env.action_space.shape[0], dtype=np.float32)
    arm_start, arm_end = 13, 23
    action[arm_start:arm_end] = np.random.uniform(-0.1, 0.1, arm_end - arm_start) * 0.3
    return action

class BaseController:
    def __init__(self, env, alpha=0.3):
        self.env = env
        self.alpha = alpha
        self._prev_action = None

    def get_action(self, obs):
        raw = heuristic_dishwipe_policy(obs, self.env)
        if self._prev_action is None:
            self._prev_action = raw.copy()
        smoothed = self.alpha * raw + (1 - self.alpha) * self._prev_action
        self._prev_action = smoothed.copy()
        return smoothed

    def reset(self):
        self._prev_action = None

class ResidualActionWrapper(gym.ActionWrapper):
    def __init__(self, env, base_controller, beta=0.5):
        super().__init__(env)
        self.base_controller = base_controller
        self.beta = beta
        self._current_obs = None

    def reset(self, **kwargs):
        self.base_controller.reset()
        result = self.env.reset(**kwargs)
        self._current_obs = result[0] if isinstance(result, tuple) else result
        return result

    def step(self, action):
        result = self.env.step(action)
        if isinstance(result, tuple):
            self._current_obs = result[0]
        return result

    def action(self, residual_action):
        base_action = self.base_controller.get_action(self._current_obs)
        final = base_action + self.beta * residual_action
        return np.clip(final, self.action_space.low, self.action_space.high)

print("✅ Helpers defined (BaseController, ResidualActionWrapper)")


## Step 4 — Train Winner on DishWipe (2M steps, RTX 5090)


In [ ]:
# Create env
train_env = gym.make(CFG["env_id"], num_envs=1, obs_mode="state",
                     control_mode=CFG["control_mode"], render_mode="rgb_array")
train_env = CPUGymWrapper(train_env)

# For Residual-SAC, wrap with ResidualActionWrapper
if winner_method == "Residual-SAC":
    with open(PROJECT_ROOT / "artifacts" / "NB07" / "best_beta.json") as f:
        beta_info = json.load(f)
    base_ctrl = BaseController(train_env, alpha=0.3)
    train_env = ResidualActionWrapper(train_env, base_ctrl, beta=beta_info["best_beta"])

# LR schedule (linear decay)
lr_sched = linear_schedule(CFG["learning_rate"], CFG["lr_end"])

# Checkpoint callback
ckpt_cb = CheckpointCallback(
    save_freq=CFG["checkpoint_freq"],
    save_path=str(ARTIFACTS_DIR / "checkpoints"),
    name_prefix=f"{winner_method.lower().replace('-', '_')}_dishwipe",
)

# Create model with RTX 5090 hyperparams
if winner_method == "PPO":
    with open(PROJECT_ROOT / "artifacts" / "NB05" / "nb05_config.json") as f:
        method_cfg = json.load(f)
    model = PPO(
        "MlpPolicy", train_env,
        learning_rate=lr_sched,
        n_steps=method_cfg.get("n_steps", 2048),
        batch_size=method_cfg.get("batch_size", 2048),
        n_epochs=method_cfg.get("n_epochs", 10),
        gamma=method_cfg.get("gamma", 0.99),
        clip_range=method_cfg.get("clip_range", 0.2),
        ent_coef=method_cfg.get("ent_coef", 0.01),
        vf_coef=method_cfg.get("vf_coef", 0.5),
        policy_kwargs={"net_arch": CFG["net_arch"],
                       "activation_fn": torch.nn.ReLU},
        verbose=1, seed=SEED, device=DEVICE,
    )
elif winner_method in ("SAC", "Residual-SAC"):
    with open(PROJECT_ROOT / "artifacts" / "NB06" / "nb06_config.json") as f:
        method_cfg = json.load(f)
    model = SAC(
        "MlpPolicy", train_env,
        learning_rate=lr_sched,
        buffer_size=method_cfg.get("buffer_size", 10_000_000),
        batch_size=method_cfg.get("batch_size", 1024),
        tau=method_cfg.get("tau", 0.005),
        gamma=method_cfg.get("gamma", 0.99),
        ent_coef=method_cfg.get("ent_coef", "auto"),
        learning_starts=method_cfg.get("learning_starts", 10_000),
        policy_kwargs={"net_arch": CFG["net_arch"],
                       "activation_fn": torch.nn.ReLU},
        verbose=1, seed=SEED, device=DEVICE,
    )
else:
    raise ValueError(f"Unknown winner: {winner_method}")

print(f"\n{'='*60}")
print(f"  Training {winner_method} on DishWipe Full-Body (RTX 5090)")
print(f"  Budget: {CFG['total_env_steps']:,} steps")
print(f"  net_arch: {CFG['net_arch']}, LR: {CFG['learning_rate']}→{CFG['lr_end']}")
print(f"{'='*60}")

# Training callback
class TrainLogCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
    def _on_step(self):
        infos = self.locals.get("infos", [])
        for info in (infos if isinstance(infos, list) else [infos]):
            if isinstance(info, dict) and "episode" in info:
                self.episode_rewards.append(float(info["episode"]["r"]))
        return True

cb = TrainLogCallback()
start_time = time.time()
model.learn(total_timesteps=CFG["total_env_steps"], callback=[cb, ckpt_cb],
            progress_bar=True)
elapsed = time.time() - start_time

model_name = f"{winner_method.lower().replace('-', '_')}_dishwipe"
model.save(str(ARTIFACTS_DIR / model_name))
train_env.close()

print(f"\nTraining done in {elapsed:.1f}s ({elapsed/3600:.1f}h)")
print(f"Model saved: {model_name}.zip")


## Step 5 — Evaluate 200 Episodes


In [ ]:
eval_env = gym.make(CFG["env_id"], num_envs=1, obs_mode="state",
                    control_mode=CFG["control_mode"], render_mode="rgb_array")
eval_env = CPUGymWrapper(eval_env)

if winner_method == "Residual-SAC":
    base_ctrl = BaseController(eval_env, alpha=0.3)
    eval_env = ResidualActionWrapper(eval_env, base_ctrl, beta=beta_info["best_beta"])

eval_results = []
for ep in range(CFG["eval_episodes"]):
    obs, info = eval_env.reset(seed=ep)
    ep_reward, ep_steps = 0.0, 0
    for step in range(CFG["max_steps_per_ep"]):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        ep_reward += float(reward)
        ep_steps += 1
        if terminated or truncated:
            break
    eval_results.append({
        "episode": ep,
        "total_reward": ep_reward,
        "steps": ep_steps,
        "success": bool(info.get("success", False)),
        "cleaned_ratio": float(info.get("cleaned_ratio", 0.0)),
    })

eval_df = pd.DataFrame(eval_results)
eval_df["method"] = winner_method
eval_df.to_csv(ARTIFACTS_DIR / "eval_200ep.csv", index=False)
eval_env.close()

print(f"Eval ({CFG['eval_episodes']} eps): "
      f"mean_reward={eval_df['total_reward'].mean():.4f}, "
      f"success={eval_df['success'].mean():.2%}, "
      f"cleaned={eval_df['cleaned_ratio'].mean():.2%}")


## Step 6 — Bootstrap CI (50K resamples) + Welch's t-test


In [ ]:
def bootstrap_ci(data, n_boot=50_000, ci=0.95):
    data = np.array(data, dtype=float)
    boot = [np.mean(np.random.choice(data, len(data), replace=True))
            for _ in range(n_boot)]
    lo = float(np.percentile(boot, (1 - ci) / 2 * 100))
    hi = float(np.percentile(boot, (1 + ci) / 2 * 100))
    return lo, hi

rewards = eval_df["total_reward"].tolist()
rew_lo, rew_hi = bootstrap_ci(rewards, CFG["bootstrap_n"])
succ = [float(s) for s in eval_df["success"]]
suc_lo, suc_hi = bootstrap_ci(succ, CFG["bootstrap_n"])

dishwipe_summary = {
    "method":              winner_method,
    "task":                "DishWipe Full-Body",
    "total_env_steps":     CFG["total_env_steps"],
    "net_arch":            CFG["net_arch"],
    "mean_reward":         float(np.mean(rewards)),
    "std_reward":          float(np.std(rewards)),
    "ci95_reward":         [rew_lo, rew_hi],
    "success_rate":        float(np.mean(succ)),
    "ci95_success":        [suc_lo, suc_hi],
    "mean_cleaned_ratio":  float(eval_df["cleaned_ratio"].mean()),
    "training_time_s":     elapsed,
}

with open(ARTIFACTS_DIR / "dishwipe_summary.json", "w") as f:
    json.dump(dishwipe_summary, f, indent=2)

print("\nDishWipe Summary:")
for k, v in dishwipe_summary.items():
    print(f"  {k}: {v}")


## Step 7 — Cross-Task Comparison (Apple vs DishWipe) + Welch's t


In [ ]:
# Load Apple eval data
apple_eval = pd.read_csv(PROJECT_ROOT / "artifacts" / "NB08" / "eval_200ep.csv")
apple_winner_data = apple_eval[apple_eval["method"] == winner_method]
apple_comp = pd.read_csv(PROJECT_ROOT / "artifacts" / "NB08" / "comparison_table.csv")
apple_winner = apple_comp[apple_comp["method"] == winner_method].iloc[0]

cross_task = pd.DataFrame([
    {"task": "Apple",    "method": winner_method,
     "mean_reward": apple_winner["mean_reward"],
     "success_rate": apple_winner["success_rate"],
     "eval_episodes": 200},
    {"task": "DishWipe", "method": winner_method,
     "mean_reward": dishwipe_summary["mean_reward"],
     "success_rate": dishwipe_summary["success_rate"],
     "eval_episodes": CFG["eval_episodes"]},
])

print("\nCross-Task Comparison:")
print(cross_task.to_string(index=False))

# Welch's t-test: Apple vs DishWipe rewards
apple_rewards = apple_winner_data["total_reward"].values
dishwipe_rewards = np.array(rewards)
t_stat, p_value = stats.ttest_ind(apple_rewards, dishwipe_rewards, equal_var=False)
# Cohen's d
nx, ny = len(apple_rewards), len(dishwipe_rewards)
pooled_sd = np.sqrt(((nx-1)*np.var(apple_rewards, ddof=1) +
                      (ny-1)*np.var(dishwipe_rewards, ddof=1)) / (nx+ny-2))
cohens_d = (np.mean(apple_rewards) - np.mean(dishwipe_rewards)) / pooled_sd if pooled_sd > 0 else 0

cross_task_stat = {
    "comparison":  "Apple vs DishWipe",
    "t_statistic": float(t_stat),
    "p_value":     float(p_value),
    "cohens_d":    float(cohens_d),
    "significant": bool(p_value < 0.05),
}
with open(ARTIFACTS_DIR / "cross_task_stat.json", "w") as f:
    json.dump(cross_task_stat, f, indent=2)

print(f"\nWelch's t-test (Apple vs DishWipe):")
print(f"  t={t_stat:.4f}, p={p_value:.4e}, Cohen's d={cohens_d:.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
tasks = ["Apple", "DishWipe"]
colors = ["#FFB74D", "#81C784"]

axes[0].bar(tasks,
            [apple_winner["mean_reward"], dishwipe_summary["mean_reward"]],
            color=colors, edgecolor="black")
axes[0].set_title(f"{winner_method}: Mean Reward by Task (200 eps)")
axes[0].set_ylabel("Mean Reward")

axes[1].bar(tasks,
            [apple_winner["success_rate"], dishwipe_summary["success_rate"]],
            color=colors, edgecolor="black")
axes[1].set_title(f"{winner_method}: Success Rate by Task")
axes[1].set_ylabel("Success Rate")
axes[1].set_ylim(0, 1)

fig.suptitle(f"NB09 — Cross-Task: {winner_method} on Apple vs DishWipe (RTX 5090)",
             fontweight="bold")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "cross_task_comparison.png", dpi=150)
plt.show()


## Learning Curve


In [ ]:
if cb.episode_rewards:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(cb.episode_rewards, alpha=0.3, color="steelblue", linewidth=0.5)
    w = min(50, len(cb.episode_rewards))
    if len(cb.episode_rewards) >= w:
        rolling = np.convolve(cb.episode_rewards, np.ones(w)/w, mode="valid")
        ax.plot(range(w-1, len(cb.episode_rewards)), rolling,
                color="darkblue", linewidth=2, label=f"Rolling avg ({w})")
    ax.set_title(f"NB09 — {winner_method} on DishWipe Training Curve (2M steps)")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(ARTIFACTS_DIR / "learning_curve.png", dpi=150)
    plt.show()


## Cleanup


In [ ]:
print("✅ NB09 Bonus DishWipe Complete (RTX 5090)")
print(f"  Method: {winner_method}")
print(f"  DishWipe mean_reward: {dishwipe_summary['mean_reward']:.4f}")
print(f"  DishWipe success_rate: {dishwipe_summary['success_rate']:.2%}")


## Artifacts

| File | Description |
|------|-------------|
| `{winner}_dishwipe.zip` | Trained model (2M steps) |
| `checkpoints/` | Periodic checkpoints (every 200K steps) |
| `learning_curve.png` | Training curve |
| `eval_200ep.csv` | 200-episode evaluation |
| `dishwipe_summary.json` | Mean reward, CI, cleaned ratio |
| `cross_task_comparison.png` | Apple vs DishWipe plot |
| `cross_task_stat.json` | Welch's t-test + Cohen's d |

## RTX 5090 Notes

- **2M steps** (was 500K) — leverages RTX 5090 throughput
- **[512, 512] ReLU** — same architecture as NB05-NB07
- **Linear LR decay** (3e-4 → 1e-5) — gradual refinement
- **200 eval episodes** (was 100) — more statistical power
- **50K bootstrap** (was 10K) — tighter CI
- **Welch's t-test + Cohen's d** — cross-task statistical comparison
- **CheckpointCallback** every 200K steps
- DishWipe-specific: `cleaned_ratio` from VirtualDirtGrid
